In [1]:
import geopandas as gpd
import pandas as pd
from shapely.geometry import shape, Polygon
import numpy as np
from pathlib import Path
import os
import sys

In [2]:
CURRENT_DIRECTORY =  os.getcwd()
PARENT_DIRECTORY = os.path.dirname(CURRENT_DIRECTORY)
print(PARENT_DIRECTORY)
BASE_DATASET_PATH = Path(PARENT_DIRECTORY).parents[0]
BASE_DATASET_PATH = BASE_DATASET_PATH / "datasets"
print(BASE_DATASET_PATH)

/Users/yitong/Desktop/Uni_Skool/Y3 Internship SingHealth/code/Mobile-AED-Study/src
/Users/yitong/Desktop/Uni_Skool/Y3 Internship SingHealth/code/Mobile-AED-Study/datasets


## Obtaining the OSM file of Singapore
Download malaysia map from here: https://download.geofabrik.de/asia/malaysia-singapore-brunei.html# -> raw directory index.

Download the .osm.pbf for 2014 to 2021 and save the files in a folder for each year (eg: 2019_geospatial)

Running in datasets/geospatial_data directory

Run this once, .poly file is used to extract singapore from the OSM file
```
curl -L -o singapore.poly "https://polygons.openstreetmap.fr/get_poly.py?id=536780"
```
Run this for each year (2014-2021)
```
cd <year>_geospatial
osmium extract -p ../singapore.poly -s complete_ways --overwrite -o singapore.osm.pbf malaysia-singapore-brunei-<year>0101.osm.pbf
```
eg: `osmium extract -p singapore.poly -s complete_ways --overwrite -o singapore.osm.pbf malaysia-singapore-brunei-190101.osm.pbf`

Convert `.osm.pbf` file to `.gpkg` file and to EPSG: 3414
```
ogr2ogr -f GPKG singapore.gpkg singapore.osm.pbf
ogr2ogr -f GPKG singapore_3414.gpkg singapore.gpkg -t_srs EPSG:3414
```
Separate lines and multipolygons into 2 different files
```
ogr2ogr -f GPKG singapore_<year>_multipolygons.gpkg singapore_3414.gpkg multipolygons
ogr2ogr -f GPKG singapore_<year>_lines.gpkg singapore_3414.gpkg lines
```
Clean up
```
rm -f singapore.osm.pbf singapore.gpkg singapore_3414.gpkg
```


In [3]:
QGIS_PATH = '/Applications/QGIS-LTR.app/Contents/Resources/python' 

# Add the QGIS Python path to the system path
sys.path.append(QGIS_PATH)
# Add path to plugins directory (usually needed for 'processing' module)
sys.path.append(os.path.join(QGIS_PATH, 'plugins'))

# QgsApplication.setPrefixPath() tells QGIS where to look for its core C++ libraries (plugins, resources, etc.).
# The first argument is the QGIS installation prefix (usually the folder containing 'bin', 'apps', etc.)
# For your Mac example, we use the path leading up to 'Resources'.
qgis_app_prefix = '/Applications/QGIS-LTR.app/Contents/Resources' # Adjust as needed

print(f"1. Setting QGIS prefix path to: {qgis_app_prefix}")
from qgis.core import QgsApplication
QgsApplication.setPrefixPath(qgis_app_prefix, True)

# Create an instance of QgsApplication. The second argument (False) means no GUI.
# This prepares the environment for non-GUI (standalone) scripting.
qgs = QgsApplication([], False) 

# Load Providers and registries (REQUIRED)
print("2. Initializing QGIS application...")
qgs.initQgis() 
print("QGIS application initialized successfully.")

1. Setting QGIS prefix path to: /Applications/QGIS-LTR.app/Contents/Resources
2. Initializing QGIS application...
QGIS application initialized successfully.


In [4]:
# Import QGIS Libraries and Processing
try:
    # Now that QGIS is initialized, we can safely import these modules
    from qgis.analysis import QgsNativeAlgorithms
    import processing
    from processing.core.Processing import Processing
    from qgis.core import QgsProject, QgsProcessingFeedback, QgsVectorLayer, QgsCoordinateReferenceSystem
    from qgis.PyQt.QtCore import QVariant 
except ImportError as e:
    print(f"\n--- FATAL ERROR: MODULE IMPORT FAILED AFTER INITIALIZATION ---")
    print(f"Please double-check the QGIS_PATH and qgis_app_prefix variables.")
    print(f"Original error: {e}")
    sys.exit(1)

In [5]:
## Uncomment to run qgis related commands
# initialise the Processing Framework
Processing.initialize()
# Add the QGIS native algorithms provider
QgsApplication.processingRegistry().addProvider(QgsNativeAlgorithms())

Logged warning: Duplicate provider native registered


False

In [6]:
def read_multipolygon_file(year, BASE_DATASET_PATH):
    """
    Reads geospatial package file

    Parameters
    ------
    year : str
        year of the geospatial data you want to read
    BASE_DATASET_PATH : pathlib.PosixPath
        the base filepath pointing to the "dataset" parent directory
    
    returns
    ------
    singapore_gpd : dataframe
        the geopackage dataframe of singapore's geospatial data
    """
    folder_name = "geospatial_data/" + str(year) + "_geospatial/singapore_" + str(year) + "_multipolygons.gpkg"
    singapore_filepath = BASE_DATASET_PATH / folder_name
    print(f"reading from: {singapore_filepath}")
    singapore_gpd = gpd.read_file(singapore_filepath)
    # print(singapore_gpd.head())
    return singapore_gpd

In [7]:
def create_manual_tags_column(df):
    """
    Create a manual_tags column to the geodataframe as the commercial value in the landuse column is not descriptive enough

    Parameters
    ------
    df : Geodataframe
        The geodataframe of interest

    Returns
    ------
    df : Geodataframe
        A modified version of the input dataframe
    """
    # add a manual_tags column
    df["manual_tags"] = None

    # commercial column is not descriptive enough. 
    # rows where commercial != null AND other_tags contains HDB will be tagged with street_shops
    # rows where commerical != null AND shop == "mall" will be tagged with mall
    singapore_landuse = df[df["landuse"] == "commercial"].copy()

    # conditions on how specific landuse will be classified
    hdb_condition = singapore_landuse["other_tags"].str.contains("HDB", na = False)
    mall_condition = singapore_landuse["shop"] == "mall"

    singapore_landuse[["other_tags", "shop"]] = singapore_landuse[["other_tags", "shop"]].astype(str)

    singapore_landuse.loc[hdb_condition, "manual_tags"] = "street_shops"
    singapore_landuse.loc[mall_condition, "manual_tags"] = "mall"

    df.update(singapore_landuse[["manual_tags"]])

    return df

    # file_path = str(BASE_DATASET_PATH / "geospatial_data/2021_geospatial/commercial_polygons.xlsx")
    # singapore_landuse.to_excel(file_path)

In [8]:
singapore_gpd = read_multipolygon_file("2018", BASE_DATASET_PATH)
singapore_gpd = create_manual_tags_column(singapore_gpd)
print("Done")

reading from: /Users/yitong/Desktop/Uni_Skool/Y3 Internship SingHealth/code/Mobile-AED-Study/datasets/geospatial_data/2018_geospatial/singapore_2018_multipolygons.gpkg
Done


#### Removing the Former Tanjong Pagar Railway Station for column building = train_station (osm_way_id = "393439098") if it is listed

In [9]:
def remove_by_osm_way_id(df, osm_way_id):
    """
    Removes a row from the Geodataframe by its osm_way_id

    Parameters
    ------
    df : Geodataframe
        Geodataframe you want to modify

    osm_way_id : str
        The osm_way_id of the identified object you want to remove

    returns
    ------
    df : Geodataframe
        Modified Geodataframe with the object removed
    """

    df = df[df["osm_way_id"] != str(osm_way_id)]

    return df

In [10]:
VALID_FILE_TYPES = {"excel", "csv", "gpkg", "xlsx"}

def save_to_file(df, file_type, filepath):
    """
    Save files to a specified location

    Parameters
    ------
    df : Dataframe
        Dataframe that you want to save

    file_type : str
        The type of file you want to save as (csv, excel, xlsx, gpkg)
    filepath : str
        Location to save the file
    """

    type = str(file_type).lower()
    
    if type not in VALID_FILE_TYPES:
        raise ValueError(f"results: status must be one of {VALID_FILE_TYPES}.")
    
    if type in ["excel", "xlsx"]:
        df.to_excel(filepath)
    
    elif type in ["csv"]:
        df.to_csv(filepath)
    
    else:
        df.to_file(filepath)
    print("Saving to filepath")
    print("Done")


In [11]:
## older versions of the map might not have the former train station
print("before removing train station:", singapore_gpd.shape)
singapore_gpd = remove_by_osm_way_id(singapore_gpd, "393439098")
print("after removing train station:", singapore_gpd.shape)

## REMEMBER TO change year in filename
filepath = str(BASE_DATASET_PATH / "geospatial_data/2018_geospatial/multipolygons_edited_2018.gpkg")
save_to_file(singapore_gpd, "gpkg", filepath)

before removing train station: (105986, 27)
after removing train station: (105985, 27)
Saving to filepath
Done


#### Creating a polygon of the major roads in singapore

In [12]:
def read_lines_file(year, BASE_DATASET_PATH):
    """
    Reads geospatial package file

    Parameters
    ------
    year : str
        year of the geospatial data you want to read
    BASE_DATASET_PATH : pathlib.PosixPath
        the base filepath pointing to the "dataset" parent directory
    
    returns
    ------
    singapore_gpd : dataframe
        the geopackage dataframe of singapore's geospatial data
    """

    folder_name = "geospatial_data/" + str(year) + "_geospatial/singapore_" + str(year) + "_lines.gpkg"
    singapore_filepath = BASE_DATASET_PATH / folder_name
    print(f"reading from {singapore_filepath}")
    singapore_gpd = gpd.read_file(singapore_filepath)
    # print(singapore_gpd.head())
    return singapore_gpd

In [13]:
singapore_lines_gpd = read_lines_file("2018", BASE_DATASET_PATH)

reading from /Users/yitong/Desktop/Uni_Skool/Y3 Internship SingHealth/code/Mobile-AED-Study/datasets/geospatial_data/2018_geospatial/singapore_2018_lines.gpkg


In [ ]:

# print(f"{singapore_2021_lines.columns} \n")
# print(singapore_2021_lines['highway'].unique())

Index(['osm_id', 'name', 'highway', 'waterway', 'aerialway', 'barrier',
       'man_made', 'z_order', 'other_tags', 'geometry'],
      dtype='object') 

['primary' 'residential' 'tertiary' 'footway' 'service' 'secondary'
 'motorway' 'motorway_link' 'trunk' None 'trunk_link' 'primary_link'
 'secondary_link' 'living_street' 'construction' 'pedestrian'
 'unclassified' 'proposed' 'steps' 'track' 'cycleway' 'path'
 'tertiary_link' 'bridleway' 'raceway' 'corridor' 'elevator' 'rest_area']


#### Based on the wiki, I have decided to include motorway, trunk, primary, secondary as the major roads in singapore to include
https://wiki.openstreetmap.org/wiki/WikiProject_Singapore#Roads

In [14]:
def get_major_roads(df, major_roads):
    """
    Get the motorway, trunk, primary and secondary roads of Singapore

    Parameter
    ------
    df : Geodataframe
        Geodataframe containing line layer information

    major_roads : list
        List of road types to be included

    Returns
    ------
    combined_major_roads : Geodataframe
        Geodataframe that containing combined information all the roads of Singapore in one entry
    """

    singapore_major_roads = df[
        df["highway"].isin(major_roads)
    ].copy()

    combined_major_roads = singapore_major_roads.dissolve()

    drop_columns = ["osm_id", "name", "highway", "waterway", "aerialway", "barrier", "man_made", "z_order", "other_tags"]
    combined_major_roads = combined_major_roads.drop(drop_columns, axis = 1, errors = 'ignore')
    combined_major_roads["major_roads"] = "yes"

    return combined_major_roads

In [15]:
# https://wiki.openstreetmap.org/wiki/WikiProject_Singapore#Roads
major_roads = [
    'motorway', 
    'trunk', 
    'primary', 
    'secondary'
]

combined_major_roads = get_major_roads(singapore_lines_gpd, major_roads)
filepath = BASE_DATASET_PATH / "geospatial_data/2018_geospatial/singapore_combined_major_roads_2018.gpkg"
save_to_file(combined_major_roads, "gpkg", filepath)


Saving to filepath
Done


#### Combine the lines of major roads into one singular line
There will be feature loss but the aim was to create a polygon of the major roads. There wasnt a need to preserve the road's features.

In [16]:
# transforming the lines into polygon using the buffer function in qgis
# processing.run("native:buffer", {'INPUT':'/Users/yitong/Desktop/Uni_Skool/Y3 Internship SingHealth/code/Mobile-AED-Study/datasets/geospatial_data/2021_geospatial/singapore_combined_major_roads_2021.gpkg|layername=singapore_combined_major_roads_2021','DISTANCE':5,'SEGMENTS':5,'END_CAP_STYLE':0,'JOIN_STYLE':0,'MITER_LIMIT':2,'DISSOLVE':True,'SEPARATE_DISJOINT':False,'OUTPUT':'/Users/yitong/Desktop/Uni_Skool/Y3 Internship SingHealth/code/Mobile-AED-Study/datasets/geospatial_data/2021_geospatial/2021_combined_roads_polygon.gpkg'})

def transform_lines_to_polygon(year: str):
    input_path = (
        "/Users/yitong/Desktop/Uni_Skool/Y3 Internship SingHealth/code/Mobile-AED-Study/"
        f"datasets/geospatial_data/{year}_geospatial/"
        f"singapore_combined_major_roads_{year}.gpkg|layername=singapore_combined_major_roads_{year}"
    )

    output_path = (
        "/Users/yitong/Desktop/Uni_Skool/Y3 Internship SingHealth/code/Mobile-AED-Study/"
        f"datasets/geospatial_data/{year}_geospatial/"
        f"combined_roads_polygon_{year}.gpkg"
    )

    buffer_params = {
        "INPUT": input_path,
        "DISTANCE": 5,
        "SEGMENTS": 5,
        "END_CAP_STYLE": 0,       # 0 = Round, 1 = Flat, 2 = Square
        "JOIN_STYLE": 0,          # 0 = Round, 1 = Miter, 2 = Bevel
        "MITER_LIMIT": 2,
        "DISSOLVE": True,         # Merge overlapping buffers
        "SEPARATE_DISJOINT": False,
        "OUTPUT": output_path,
    }

    processing.run("native:buffer", buffer_params)

    print("Done")

In [17]:
transform_lines_to_polygon("2018")

Done


### Creating a grid over Singapore using qgis (only need to run once)
The grid size will be a hectare large ($100m \times 100m$)

The extents of the grid was determined by looking at Google maps to find the boundaries of Singapore. And converting to EPSG: 3414

In [ ]:
# processing.run("native:creategrid", {'TYPE':2,'EXTENT':'2666.773500000,56466.773500000,15657.950600000,50257.950600000 [EPSG:3414]','HSPACING':100,'VSPACING':100,'HOVERLAY':0,'VOVERLAY':0,'CRS':QgsCoordinateReferenceSystem('EPSG:3414'),'OUTPUT':'/Users/yitong/Desktop/Uni_Skool/Y3 Internship SingHealth/code/Mobile-AED-Study/datasets/geospatial_data/2021_geospatial/hectare_grid.gpkg'})

output_path = (
    str(BASE_DATASET_PATH) + "/geospatial_data/hectare_grid.gpkg"
)

grid_params = {
    "TYPE": 2,
    "EXTENT": "2666.773500000,56466.773500000,15657.950600000,50257.950600000 [EPSG:3414]",
    "HSPACING": 100,
    "VSPACING": 100,
    "HOVERLAY": 0,
    "VOVERLAY": 0,
    "CRS": QgsCoordinateReferenceSystem("EPSG:3414"),
    "OUTPUT": output_path,
}

processing.run("native:creategrid", grid_params)

{'OUTPUT': '/Users/yitong/Desktop/Uni_Skool/Y3 Internship SingHealth/code/Mobile-AED-Study/datasets/geospatial_data/hectare_grid.gpkg'}

#### Adding the polygons to postgres
Command will need to be ran manually on command line/terminal

In [18]:
def construct_add_command(filename: str, database_name: str, username: str):

    cmd = [f"ogr2ogr -f PostgreSQL PG:\"dbname={database_name} user={username}\" ", 
           filename + ".gpkg",
           " -nln ", filename,
           " -lco GEOMETRY_NAME=geom ",
           "-lco FID=id ",
           "-nlt PROMOTE_TO_MULTI ",
           filename
    ]
    result = "".join(cmd)
    return result


def add_to_postgres(database_name: str, username: str, year: int | None = None):
    """
    adds the layers from .gpkg file to postgreSQL
    Can't get PostgreSQL command to work,
    need to manually copy command output and run in terminal
    """
    
    roads_filename = "combined_roads_polygon" + "_" + str(year)
    cmd_1 = construct_add_command(roads_filename, database_name, username)
    print(cmd_1)

    multipolygon_filename = "multipolygons_edited" + "_" + str(year)
    cmd_2 = construct_add_command(multipolygon_filename, database_name, username)
    print(cmd_2)

    cmd_indexes = f"""
CREATE INDEX IF NOT EXISTS hectare_grid_geom_idx ON hectare_grid USING GIST (geom);

CREATE INDEX IF NOT EXISTS multipolygons_geom_idx ON {roads_filename} USING GIST (geom);

CREATE INDEX IF NOT EXISTS roads_polygon_geom_idx ON {multipolygon_filename} USING GIST (geom);
    """
    print(cmd_indexes)



In [19]:
add_to_postgres("postgis", "yitong", 2018)

ogr2ogr -f PostgreSQL PG:"dbname=postgis user=yitong" combined_roads_polygon_2018.gpkg -nln combined_roads_polygon_2018 -lco GEOMETRY_NAME=geom -lco FID=id -nlt PROMOTE_TO_MULTI combined_roads_polygon_2018
ogr2ogr -f PostgreSQL PG:"dbname=postgis user=yitong" multipolygons_edited_2018.gpkg -nln multipolygons_edited_2018 -lco GEOMETRY_NAME=geom -lco FID=id -nlt PROMOTE_TO_MULTI multipolygons_edited_2018

CREATE INDEX IF NOT EXISTS hectare_grid_geom_idx ON hectare_grid USING GIST (geom);

CREATE INDEX IF NOT EXISTS multipolygons_geom_idx ON combined_roads_polygon_2018 USING GIST (geom);

CREATE INDEX IF NOT EXISTS roads_polygon_geom_idx ON multipolygons_edited_2018 USING GIST (geom);
    


#### Fixing invalid geometries in multipolygon table

In [20]:
def fix_invalid_geometries(table_name: str):
    cmd = f"""UPDATE {table_name}
        SET geom = ST_CollectionExtract(ST_MakeValid(geom), 3)::geometry(MultiPolygon, 3414)
        WHERE NOT ST_IsValid(geom);
        """
    
    print(cmd)

fix_invalid_geometries("multipolygons_edited_2018")

UPDATE multipolygons_edited_2018
        SET geom = ST_CollectionExtract(ST_MakeValid(geom), 3)::geometry(MultiPolygon, 3414)
        WHERE NOT ST_IsValid(geom);
        


#### Adding spatial index for faster spatial joins

#### Intersecting multipolygons with hectare grid

In [21]:
def multipolygon_grids(output_table_name: str, input_table_name: str):
    cmd = f"""
DROP TABLE IF EXISTS {output_table_name};

CREATE TABLE {output_table_name} AS
SELECT 
    g.id AS grid_id,
    g.row_index as grid_row,
    g.col_index as grid_col,
    mp.id AS id,
    mp.osm_id as osm_id,
    mp.osm_way_id as osm_way_id,
    mp.name AS name,
	mp.amenity as amenity,
	mp.building as building,
	mp.landuse as landuse,
	mp.military as military,
	mp.office as office,
	mp.place as place,
	mp.shop as shop,
	mp.other_tags as other_tags,
  mp.manual_tags as manual_tags,
    ST_Intersection(ST_MakeValid(mp.geom), g.geom) AS geom
FROM hectare_grid g
JOIN {input_table_name} mp
  ON ST_Intersects(g.geom, mp.geom);
    """

    print(cmd)

multipolygon_grids("polygons_grid_2018", "multipolygons_edited_2018")


DROP TABLE IF EXISTS polygons_grid_2018;

CREATE TABLE polygons_grid_2018 AS
SELECT 
    g.id AS grid_id,
    g.row_index as grid_row,
    g.col_index as grid_col,
    mp.id AS id,
    mp.osm_id as osm_id,
    mp.osm_way_id as osm_way_id,
    mp.name AS name,
	mp.amenity as amenity,
	mp.building as building,
	mp.landuse as landuse,
	mp.military as military,
	mp.office as office,
	mp.place as place,
	mp.shop as shop,
	mp.other_tags as other_tags,
  mp.manual_tags as manual_tags,
    ST_Intersection(ST_MakeValid(mp.geom), g.geom) AS geom
FROM hectare_grid g
JOIN multipolygons_edited_2018 mp
  ON ST_Intersects(g.geom, mp.geom);
    


#### Intersecting road polygons with hectare grid

In [22]:
def road_polygon_grid(output_table_name: str, input_table_name: str):
    cmd = f"""
    DROP TABLE IF EXISTS {output_table_name};

    Create TABLE {output_table_name} AS
    SELECT
    g.id AS grid_id,
    g.row_index as grid_row,
    g.col_index as grid_col,
    mp.major_roads as major_roads,
    ST_Intersection(ST_MakeValid(mp.geom), g.geom) AS geom
    FROM hectare_grid g
    JOIN {input_table_name} mp
    ON ST_Intersects(g.geom, mp.geom);
    """
    print(cmd)

road_polygon_grid("roads_grid_2018", "combined_roads_polygon_2018")


    DROP TABLE IF EXISTS roads_grid_2018;

    Create TABLE roads_grid_2018 AS
    SELECT
    g.id AS grid_id,
    g.row_index as grid_row,
    g.col_index as grid_col,
    mp.major_roads as major_roads,
    ST_Intersection(ST_MakeValid(mp.geom), g.geom) AS geom
    FROM hectare_grid g
    JOIN combined_roads_polygon_2018 mp
    ON ST_Intersects(g.geom, mp.geom);
    


#### Combining the 2 tables

In [23]:
def combine_table(output_table: str, multipolygon_table: str, roads_table: str):
    cmd = f"""
    DROP TABLE IF EXISTS {output_table};
    CREATE TABLE {output_table} AS
    SELECT 
        grid_id, grid_row, grid_col, geom, major_roads,
        NULL::integer AS id, NULL::varchar AS osm_way_id,
        NULL::varchar AS name, NULL::varchar AS amenity,
        NULL::varchar AS building, NULL::varchar AS landuse,
        NULL::varchar AS military, NULL::varchar AS office,
        NULL::varchar AS place, NULL::varchar AS shop,
        NULL::varchar AS other_tags, NULL::varchar AS manual_tags,
        NULL::varchar AS osm_id
    FROM {roads_table}

    UNION ALL

    SELECT 
        grid_id, grid_row, grid_col, geom,
        NULL::text AS major_roads,
        id, osm_way_id, name, amenity,
        building, landuse, military,
        office, place, shop, other_tags,
        manual_tags, osm_id
    FROM {multipolygon_table};
    """
    
    print(cmd)

combine_table("combined_grid_2018", "polygons_grid_2018", "roads_grid_2018")


    DROP TABLE IF EXISTS combined_grid_2018;
    CREATE TABLE combined_grid_2018 AS
    SELECT 
        grid_id, grid_row, grid_col, geom, major_roads,
        NULL::integer AS id, NULL::varchar AS osm_way_id,
        NULL::varchar AS name, NULL::varchar AS amenity,
        NULL::varchar AS building, NULL::varchar AS landuse,
        NULL::varchar AS military, NULL::varchar AS office,
        NULL::varchar AS place, NULL::varchar AS shop,
        NULL::varchar AS other_tags, NULL::varchar AS manual_tags,
        NULL::varchar AS osm_id
    FROM roads_grid_2018

    UNION ALL

    SELECT 
        grid_id, grid_row, grid_col, geom,
        NULL::text AS major_roads,
        id, osm_way_id, name, amenity,
        building, landuse, military,
        office, place, shop, other_tags,
        manual_tags, osm_id
    FROM polygons_grid_2018;
    


#### Extract the table to `.gpkg` file

In [27]:
def extract_table_into_file(database_name: str, username: str, table_name: str):
    cmd = f"""
ogr2ogr -f GPKG {table_name}.gpkg PG:"dbname={database_name} user={username}" {table_name}
    """
    print(cmd)

extract_table_into_file("postgis", "yitong", "combined_grid_2018")



ogr2ogr -f GPKG combined_grid_2018.gpkg PG:"dbname=postgis user=yitong" combined_grid_2018
    


In [75]:
filepath = BASE_DATASET_PATH / "geospatial_data/2018_geospatial/combined_grid_2018.gpkg"
combined_grid = gpd.read_file(filepath)
# sort by ascending grid_id
combined_grid = combined_grid.sort_values(by = "grid_id").copy()
print("Done")

Done


In [66]:
# combined_grid["grid_id"]

shorten_df = combined_grid[
    (combined_grid["grid_id"] >= 98430) &
    (combined_grid["grid_id"] <= 98430)].copy()

# sort by ascending grid_id
shorten_df = shorten_df.sort_values(by="grid_id").copy()
shorten_df

,grid_id,grid_row,grid_col,major_roads,id,osm_way_id,name,amenity,building,landuse,military,office,place,shop,other_tags,manual_tags,osm_id,geometry
412749,98430,165,284,None,99439.0,477223917,None,None,residential,None,None,None,None,None,"""addr:city""=>""Singapore"",""addr:housenumber""=>""...",None,None,"POLYGON ((31162.155 33691.045, 31166.774 33687..."
412743,98430,165,284,None,99438.0,477223916,None,None,residential,None,None,None,None,None,"""addr:city""=>""Singapore"",""addr:housenumber""=>""...",None,None,"POLYGON ((31135.701 33716.986, 31149.1 33706.5..."
412737,98430,165,284,None,99437.0,477223915,None,None,residential,None,None,None,None,None,"""addr:city""=>""Singapore"",""addr:housenumber""=>""...",None,None,"MULTIPOLYGON (((31094.49 33734.821, 31113.476 ..."
39404,98430,165,284,None,383.0,None,Central,None,None,None,None,None,None,None,"""ISO3166-2""=>""SG-01"",""wikidata""=>""Q2544592""",None,3831712,"POLYGON ((31166.774 33757.951, 31166.774 33657..."
111043,98430,165,284,None,1713.0,45249369,Sungei Whampoa,None,None,None,None,None,None,None,"""water""=>""river""",None,None,"POLYGON ((31087.81 33757.951, 31157.204 33757...."
442304,98430,165,284,None,47.0,None,Singapore,None,None,None,None,None,island,None,"""int_name""=>""Singapore"",""name:ace""=>""Singapura...",None,1769123,"POLYGON ((31166.774 33757.951, 31166.774 33657..."


In [76]:
# to keep track of the polygon areas of each hectare
hectare_objects = {"airport": 0, "mall": 0, "school": 0, "public_transport_station": 0,
             "major_road": 0, "residential": 0, "workforce": 0, "street_shops": 0}

CHANGI_AIRPORT = "174037464"
SELETAR_AIRPORT = "381760619"
AIRPORT = "airport"
MALL = "mall"
MAJOR_ROAD = "major_road"
STREET_SHOP = "street_shops"
SCHOOL = "school"
RESIDENTIAL = "residential"
WORKFORCE = "workforce"

In [80]:
def calculate_area(row, temp_dict, classification):
    if classification == AIRPORT:
        # if the hectare contains airport, that hectare will be classified as airport
        # so make the airport area be the largest
        temp_dict[AIRPORT] += 10000.0
        return temp_dict
    
    if classification == MALL:
        temp_dict[MALL] += 10000.0
        return temp_dict
    
    # if not airport or mall, add up the area of that classification
    temp_dict[classification] += row.geometry.area
    return temp_dict

def decide_classification(current_grid_id, temp_dict, results_pd):
    if all(value == 0 for value in temp_dict.values()):
        classification = None

    else:
        # select the classification with the biggest area
        classification = max(temp_dict, key = temp_dict.get)

    # print(current_grid_id, temp_dict)

    grid_id = current_grid_id[0]
    grid_row = current_grid_id[1]
    grid_col = current_grid_id[2]

    # add to the dataframe
    results_pd.loc[len(results_pd)] = [grid_id, grid_row, grid_col, classification]
    

In [81]:
def classify_hectare(input_pd, results_pd):

    current_grid_id = [0, 0, 0]
    temp_dict = hectare_objects.copy()

    for r in input_pd.itertuples(index=False):
        # if the grid_id incremented, means we moving onto the next hectare grid
        if current_grid_id[0] != r.grid_id and current_grid_id[0] != 0:
            decide_classification(current_grid_id, temp_dict, results_pd)

            # reset temp_dict
            temp_dict = hectare_objects.copy()

        # keep track of the current grid id
        current_grid_id = [r.grid_id, r.grid_row, r.grid_col]

        # if the hectare contains airport polygons
        if r.osm_way_id == CHANGI_AIRPORT or r.osm_way_id == SELETAR_AIRPORT:
            temp_dict = calculate_area(r, temp_dict, AIRPORT)
        
        ## As I would want the manual_tags column to override 
        ## the classification from the other columns (mainly for malls and street shops),
        ## they will be checked for first

        # check for malls
        if r.manual_tags == MALL or r.shop == MALL or r.shop == "supermarket" or r.landuse == "retail":
            temp_dict = calculate_area(r, temp_dict, MALL)
            # move to next iteration, manual_tags column should override the classification decision for that row
            continue
        
        # check for street shops
        if r.manual_tags == STREET_SHOP:
            temp_dict = calculate_area(r, temp_dict, STREET_SHOP)
            # move to next iteration, manual_tags column should override the classification decision for that row
            continue

        # check for major roads. major_roads column will be "yes"
        if r.major_roads == "yes":
            temp_dict = calculate_area(r, temp_dict, MAJOR_ROAD)
        
        # check for schools  
        # (primary, secondary, Junior College, polytechnic, universities, childcare)
        if str(r.amenity).lower() in (SCHOOL, "university", "college", "childcare"):
            temp_dict = calculate_area(r, temp_dict, SCHOOL)

        # check for residential area (HDB, condo, landed)
        if r.landuse == RESIDENTIAL or r.place == "neighbourhood" or r.building == RESIDENTIAL:
            temp_dict = calculate_area(r, temp_dict, RESIDENTIAL)

        # check for workforce
        if str(r.building).lower() in ["industrial", "construction"] or r.office is not None or r.landuse == "industrial":
            temp_dict = calculate_area(r, temp_dict, WORKFORCE)

    # classify the last hectare manually
    decide_classification(current_grid_id, temp_dict, results_pd)

    return results_pd 


## try to optimise classify_hectare() method
And see if `r.office is not None` will cover all NULL, nan and None

In [82]:
columns_list = ["grid_id", "grid_row", "grid_col", "hectare_classification"]
# for working with classifying of each hectare
results_pd = pd.DataFrame(columns = columns_list)

output_pd = classify_hectare(combined_grid, results_pd)
# output_pd = classify_hectare(shorten_df, results_pd)
output_pd.shape

(133661, 4)

In [83]:
# filepath = BASE_DATASET_PATH / "geospatial_data/2018_geospatial/grid_classification.xlsx"
filepath = BASE_DATASET_PATH / "geospatial_data/2018_geospatial/grid_classification.xlsx"
output_pd.to_excel(filepath)

#### Cleaning up

In [85]:
def cleaning_up(year: str):
    cmd = f"""
    rm -f multipolygons_edited_{year}.gpkg singapore_{year}_lines.gpkg singapore_{year}_multipolygons.gpkg singapore_combined_major_roads_{year}.gpkg
"""
    print(cmd)

cleaning_up("2017")
    


    rm -f multipolygons_edited_2017.gpkg singapore_2017_lines.gpkg singapore_2017_multipolygons.gpkg singapore_combined_major_roads_2017.gpkg



In [ ]:
# columns_list = ["grid_id", "grid_row", "grid_col", "hectare_classification"]
# # for working with classifying of each hectare
# results_pd = pd.DataFrame(columns = columns_list)

# output_pd = classify_hectare(combined_2021, results_pd)
# output_pd.shape

(286883, 4)

In [ ]:
# filepath = BASE_DATASET_PATH / "geospatial_data/2021_geospatial/result_classification.xlsx"
# output_pd.to_excel(filepath)

In [ ]:
# # for r in shorten_combined_2021.itertuples(index=False):
# #     print(r.landuse)
# for r in shorten_combined_2021.itertuples(index=False):
#     # print(r)
#     print(r.building, type(r.building))
#     # print(r.office, type(r.office))